In [0]:
#pip install ortools

In [0]:
#%restart_python

In [0]:
import time
import numpy as np
import pandas as pd
from scipy.sparse.csgraph import minimum_spanning_tree
from ortools.constraint_solver import pywrapcp, routing_enums_pb2 
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

# graph
nodes = (spark.table("workspace.spb.silver_walk_nodes")
         .select("node_id", "node_idx", "lat", "lon").toPandas())
edges = (spark.table("workspace.spb.silver_walk_edges")
         .select("src_idx", "dst_idx", "length_m").toPandas())


# distance matrix in memory, in node_id order
pdf = spark.table("workspace.spb.gold_node_distances").toPandas()
M_df = pdf.pivot(index="src_node_id", columns="dst_node_id", values="dist_m")
ids = M_df.index.to_numpy()
M = M_df.to_numpy()
n = len(M)

N = len(nodes)
assert nodes.node_idx.max() == N - 1

graph = csr_matrix(
    (edges.length_m.to_numpy(),
     (edges.src_idx.to_numpy(), edges.dst_idx.to_numpy())),
    shape=(N, N),
)

# graph index of each stop, in matrix order
targets = nodes.set_index("node_id").loc[ids, "node_idx"].to_numpy()
print(N, "graph nodes |", n, "stops |", len(targets), "targets")

print(type(M), M.shape)

In [0]:



def tour_length(M, order):
    order = np.asarray(order)
    return M[order, np.roll(order, -1)].sum()


def nearest_neighbor(M, start=0):
    n = len(M)
    visited = np.zeros(n, dtype=bool)
    order = [start]
    visited[start] = True
    for _ in range(n - 1):
        d = M[order[-1]].copy()      # copy of M
        d[visited] = np.inf          # can't revisit
        nxt = int(np.argmin(d))
        visited[nxt] = True
        order.append(nxt)
    return order


def best_nearest_neighbor(M):
    best, best_len = None, np.inf
    for s in range(len(M)):
        o = nearest_neighbor(M, s)
        L = tour_length(M, o)
        if L < best_len:
            best, best_len = o, L
    return best


def solve_ortools(M, time_limit_s=60, initial_order=None): # CLAUDE
    n = len(M)
    D = np.rint(M).astype(np.int64)

    manager = pywrapcp.RoutingIndexManager(n, 1, 0)    
    routing = pywrapcp.RoutingModel(manager)

    def dist_cb(i, j):
        return int(D[manager.IndexToNode(i), manager.IndexToNode(j)])

    transit = routing.RegisterTransitCallback(dist_cb)
    routing.SetArcCostEvaluatorOfAllVehicles(transit)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.seconds = time_limit_s

    if initial_order is not None:
        # warm start: rotate so the tour begins at the depot (0), then hand over the other stops
        k = list(initial_order).index(0)
        rotated = list(initial_order[k:]) + list(initial_order[:k])
        routing.CloseModelWithParameters(params)
        start = routing.ReadAssignmentFromRoutes([rotated[1:]], True)
        solution = routing.SolveFromAssignmentWithParameters(start, params)
    else:
        solution = routing.SolveWithParameters(params)

    assert solution is not None, "OR-Tools found no solution"
    index = routing.Start(0)
    order = []
    while not routing.IsEnd(index):
        order.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    return order


def mst_weight(M):
    """Weight of the minimum spanning tree: a (weak) lower bound on any closed tour."""
    return minimum_spanning_tree(M).sum()

In [0]:
n = len(M)

nn = nearest_neighbor(M, 0)
assert sorted(nn) == list(range(n))                       # every stop exactly once
print("nearest neighbor, one start :", round(tour_length(M, nn) / 1000, 1), "km")

bnn = best_nearest_neighbor(M)                            # about a minute at n = 1203
print("nearest neighbor, best start:", round(tour_length(M, bnn) / 1000, 1), "km")

ot = solve_ortools(M, time_limit_s=120, initial_order=bnn)
assert sorted(ot) == list(range(n))
print("OR-Tools                    :", round(tour_length(M, ot) / 1000, 1), "km")

print("MST lower bound             :", round(mst_weight(M) / 1000, 1), "km")

In [0]:
tour_df = pd.DataFrame({
    "stop_order": np.arange(n),
    "node_id": ids[ot],
    "leg_m": M[ot, np.roll(ot, -1)],     
})
tour_df["cum_m"] = tour_df.leg_m.cumsum()
spark.createDataFrame(tour_df).write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_tour")

In [0]:



def route_paths(graph, stops_idx, batch=100):
    """
    graph      : sparse N x N matrix of edge lengths (m), each edge stored once
    stops_idx  : graph node indices of the stops, in tour order (closed loop)
    returns    : list of paths, one per leg; leg i goes stop i -> stop i+1 (last leg returns to stop 0)
    """
    stops_idx = np.asarray(stops_idx)
    n = len(stops_idx)
    legs = []
    for i0 in range(0, n, batch):
        src = stops_idx[i0:i0 + batch]
        _, pred = dijkstra(graph, directed=False, indices=src, return_predecessors=True)
        for k, a in enumerate(src):
            b = stops_idx[(i0 + k + 1) % n]
            path = [int(b)]
            while path[-1] != a:
                p = int(pred[k, path[-1]])
                if p < 0:
                    raise ValueError(f"no path from {a} to {b}")
                path.append(p)
            legs.append(path[::-1])
    return legs


def flatten_route(legs):
    seq, leg_no = [], []
    for i, path in enumerate(legs):
        part = path if i == 0 else path[1:]
        seq.extend(part)
        leg_no.extend([i] * len(part))
    return np.array(seq), np.array(leg_no)


def check_legs(graph, legs, M, tol=1e-6):
    """Each leg's summed edge lengths must equal the matrix distance for that leg."""
    G = graph.maximum(graph.T).tocsr()          # symmetric, so lookups work in either direction
    n = len(legs)
    worst = 0.0
    for i, path in enumerate(legs):
        p = np.asarray(path)
        total = float(np.asarray(G[p[:-1], p[1:]]).sum()) if len(p) > 1 else 0.0
        worst = max(worst, abs(total - M[i, (i + 1) % n]))
    return worst

In [0]:
targets = nodes.set_index("node_id").loc[ids, "node_idx"].to_numpy()
print(len(targets))        # 1203

In [0]:
stops_idx = targets[ot]                      # graph node_idx of each stop, in tour order
legs = route_paths(graph, stops_idx)         # takes about the same time as the matrix: ~1.5 min

Mo = M[np.ix_(ot, ot)]                       # matrix reordered into tour order
print("worst leg mismatch (m):", check_legs(graph, legs, Mo))    # should be ~0

seq, leg_no = flatten_route(legs)
print(len(seq), "route nodes")

route = (nodes.set_index("node_idx").loc[seq, ["node_id", "lat", "lon"]]
         .reset_index(drop=True))
route.insert(0, "seq", np.arange(len(route)))
route["leg_no"] = leg_no

spark.createDataFrame(route).write.format("delta").mode("overwrite").saveAsTable("workspace.spb.gold_route_nodes")

In [0]:
import pydeck as pdk

route = spark.table("workspace.spb.gold_route_nodes").orderBy("seq").toPandas()
bars = spark.table("workspace.spb.silver_bars").select("lon", "lat", "name", "h3_cell").toPandas()

# 1. density layer: bars per H3 cell, shaded yellow to red
cells = bars.groupby("h3_cell").size().rename("n_bars").reset_index()
t = cells.n_bars / cells.n_bars.max()
cells["color"] = [[255, int(220 * (1 - x)), 0, 110] for x in t]

# 2. route: one path per leg, colored from blue (start) to red (end)
def leg_color(i, n):
    x = i / max(n - 1, 1)
    return [int(255 * x), int(80 + 100 * (1 - x)), int(255 * (1 - x))]

n_legs = route.leg_no.max() + 1
paths = [{"leg": int(leg),
          "path": g.sort_values("seq")[["lon", "lat"]].to_numpy().tolist(),
          "color": leg_color(leg, n_legs)}
         for leg, g in route.groupby("leg_no")]

deck = pdk.Deck(
    layers=[
        pdk.Layer("H3HexagonLayer", cells, get_hexagon="h3_cell", get_fill_color="color",
                  extruded=False, pickable=True),
        pdk.Layer("PathLayer", paths, get_path="path", get_color="color",
                  width_min_pixels=2, pickable=True),
        pdk.Layer("ScatterplotLayer", bars, get_position="[lon, lat]", get_radius=25,
                  get_fill_color=[255, 255, 255, 200]),
    ],
    initial_view_state=pdk.ViewState(longitude=30.32, latitude=59.93, zoom=10),
    map_style="dark",
)

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# 1. meters walked at every route node, from the real edge lengths
G = graph.maximum(graph.T).tocsr()
step_m = np.asarray(G[seq[:-1], seq[1:]]).ravel()
cum_m = np.r_[0.0, np.cumsum(step_m)]
assert abs(cum_m[-1] - tour_length(M, ot)) < 1e-3     # must equal the tour length

route = route.sort_values("seq").reset_index(drop=True)
assert len(route) == len(cum_m)
route["cum_m"] = cum_m

# 2. meters walked when each stop is reached (stop 0 at the start, stop i at the end of leg i-1)
leg_end_cum = route.groupby("leg_no").cum_m.max().sort_index().to_numpy()
stop_arrive = pd.DataFrame({"node_id": ids[ot], "arrive_m": np.r_[0.0, leg_end_cum[:-1]]})

# 3. attach that to every bar
bar_nodes_pd = spark.table("workspace.spb.snapped_walk_nodes").select("bar_id", "node_id").toPandas()
bars_pd = (spark.table("workspace.spb.silver_bars")
           .select(F.col("id").alias("bar_id"), "lon", "lat", "name").toPandas())
bars_live = bars_pd.merge(bar_nodes_pd, on="bar_id").merge(stop_arrive, on="node_id")
print(len(bars_live), "bars")                         # should be 1363

In [0]:
"""
make_live_map(route, bars) -> a self-contained HTML string: an animated map that draws the tour leg by leg
with a live distance counter, bars lighting up as they are reached, a "next stop" panel,
play/pause, speed and scrub controls. This is all vibe coded but works out well.

route : pandas DataFrame with columns  seq, lon, lat, leg_no, cum_m   (cum_m = metres walked so far, increasing)
bars  : pandas DataFrame with columns  lon, lat, name, arrive_m       (arrive_m = metres walked when the bar is reached)
"""
import json
import numpy as np

LEAFLET_CSS = "https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"
LEAFLET_JS = "https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"

STRINGS = {
    "en": {
        "locale": "en-US", "title": "St. Petersburg bar crawl",
        "kmWalked": "km walked", "of": "of", "kmUnit": "km", "atPace": "at 5 km/h",
        "barsReached": "bars reached", "nextStop": "Next stop",
        "restart": "Restart", "speed": "Speed", "follow": "Follow the walker",
        "play": "Play", "pause": "Pause", "replay": "Replay", "progress": "Progress",
        "nextDist": "{d} away", "toGo": "{d} to go", "backToStart": "Back to the start", "complete": "Tour complete",
        "unnamed": "(unnamed)", "more": "+{n} more", "m": "m", "km": "km", "days": "d", "hours": "h",
        "speedLabel": "{s} km per second (about {n} s for the whole tour)",
        "zoomIn": "Zoom in", "zoomOut": "Zoom out", "expand": "Show details", "collapse": "Hide details",
    },
    "ru": {
        "locale": "ru-RU", "title": "Барный маршрут по Санкт-Петербургу",
        "kmWalked": "км пройдено", "of": "из", "kmUnit": "км", "atPace": "при 5 км/ч",
        "barsReached": "баров посещено", "nextStop": "Следующая остановка",
        "restart": "Сначала", "speed": "Скорость", "follow": "Следить за путником",
        "play": "Старт", "pause": "Пауза", "replay": "Заново", "progress": "Прогресс",
        "nextDist": "ещё {d}", "toGo": "осталось {d}", "backToStart": "Назад к старту", "complete": "Маршрут пройден",
        "unnamed": "(без названия)", "more": "и ещё {n}", "m": "м", "km": "км", "days": "дн.", "hours": "ч",
        "speedLabel": "{s} км/с (весь маршрут примерно за {n} с)",
        "zoomIn": "Приблизить", "zoomOut": "Отдалить", "expand": "Показать подробности", "collapse": "Скрыть подробности",
    },
}

_TEMPLATE = r"""<!doctype html>
<html lang="__LANG__">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<link rel="stylesheet" href="__LEAFLET_CSS__">
<style>
  html, body { margin: 0; height: 100%; background: #0e1116; font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; }
  #map { position: absolute; inset: 0; background: #0e1116; }
  .panel { position: absolute; z-index: 1000; left: 12px; top: 12px; width: min(300px, calc(100vw - 24px));
           box-sizing: border-box; background: rgba(14,17,22,.88); color: #e8eaed; border: 1px solid rgba(255,255,255,.14);
           border-radius: 12px; padding: 14px 16px; backdrop-filter: blur(6px); }
  .head { display: flex; align-items: center; justify-content: space-between; gap: 8px; margin-bottom: 8px; }
  .panel h1 { font-size: 13px; font-weight: 600; letter-spacing: .04em; text-transform: uppercase; color: #9aa4b2; margin: 0; min-width: 0; }
  #toggle { padding: 5px 8px; line-height: 0; flex: none; }
  #toggle svg { transform: rotate(180deg); }
  .compact #toggle svg { transform: none; }
  .compact .detail, .compact .sub { display: none; }
  .big { font-size: 38px; font-weight: 700; line-height: 1; font-variant-numeric: tabular-nums; }
  .big small { font-size: 16px; font-weight: 500; color: #9aa4b2; margin-left: 4px; }
  .sub { font-size: 13px; color: #9aa4b2; margin-top: 4px; font-variant-numeric: tabular-nums; }
  .row { display: flex; justify-content: space-between; gap: 10px; margin-top: 12px; font-size: 13px; }
  .row b { display: block; font-size: 18px; font-variant-numeric: tabular-nums; color: #e8eaed; }
  .row span { color: #9aa4b2; }
  .next { margin-top: 12px; padding-top: 10px; border-top: 1px solid rgba(255,255,255,.12); font-size: 13px; }
  .next span { color: #9aa4b2; }
  .next b { display: block; font-size: 16px; color: #ffd166; margin: 2px 0; overflow-wrap: anywhere; }
  .controls { display: flex; gap: 8px; margin-top: 14px; align-items: center; }
  button { background: #2a3140; color: #e8eaed; border: 1px solid rgba(255,255,255,.18); border-radius: 8px;
           padding: 7px 12px; font-size: 13px; cursor: pointer; }
  button:hover { background: #374056; }
  button.primary { background: #3b82f6; border-color: #3b82f6; min-width: 76px; }
  input[type=range] { width: 100%; margin: 10px 0 0; accent-color: #3b82f6; }
  label { font-size: 12px; color: #9aa4b2; display: block; margin-top: 10px; }
  .follow { display: flex; align-items: center; gap: 6px; margin-top: 10px; font-size: 12px; color: #9aa4b2; }
  .leaflet-container { background: #0e1116; }
  @media (max-width: 640px) {
    .panel { left: 8px; right: 8px; top: 8px; width: auto; padding: 8px 12px; background: rgba(14,17,22,.72); }
    #toggle { position: absolute; top: 8px; right: 8px; }
    .head { margin-bottom: 4px; padding-right: 40px; }
    .head { margin-bottom: 2px; }
    .panel h1 { font-size: 11px; }
    #toggle { top: 6px; right: 6px; padding: 3px 7px; }
    .big { font-size: 26px; }
    .big small { font-size: 13px; }
    .next b { font-size: 15px; margin: 1px 0; }
    button { padding: 6px 10px; }
    .row { margin-top: 8px; }
    .next { margin-top: 6px; padding-top: 6px; }
    .controls { margin-top: 8px; }
    input[type=range] { margin-top: 8px; }
  }
</style>
</head>
<body>
<div id="map"></div>
<div class="panel" id="panel">
  <div class="head">
    <h1 id="title"></h1>
    <button id="toggle" aria-expanded="true"><svg width="14" height="14" viewBox="0 0 14 14" aria-hidden="true"><path d="M2 5l5 5 5-5" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/></svg></button>
  </div>
  <div class="big"><span id="km">0</span><small data-t="kmWalked"></small></div>
  <div class="sub"><span id="pct">0</span>% <span data-t="of"></span> <span id="totalkm"></span> <span data-t="kmUnit"></span> &middot; <span id="time"></span> <span data-t="atPace"></span></div>
  <div class="row detail">
    <div><b><span id="visited">0</span> / <span id="nbars"></span></b><span data-t="barsReached"></span></div>
  </div>
  <div class="next"><span data-t="nextStop"></span><b id="nextname">&nbsp;</b><span id="nextdist">&nbsp;</span></div>
  <input type="range" id="scrub" class="detail" min="0" max="1000" value="0">
  <div class="controls">
    <button class="primary" id="play"></button>
    <button id="restart" data-t="restart"></button>
  </div>
  <div class="detail">
    <label><span data-t="speed"></span>: <span id="speedlabel"></span>
      <input type="range" id="speed" min="0" max="150" value="50">
    </label>
    <div class="follow"><input type="checkbox" id="follow"><label for="follow" style="margin:0" data-t="follow"></label></div>
  </div>
</div>
<script id="data" type="application/json">__DATA__</script>
<script src="__LEAFLET_JS__"></script>
<script>
(function () {
  var D = JSON.parse(document.getElementById('data').textContent);
  var T = D.t;
  var lon = D.route.lon, lat = D.route.lat, cum = D.route.cum, legEnd = D.route.legEnd;
  var nLegs = legEnd.length, total = cum[cum.length - 1];
  var legEndCum = legEnd.map(function (i) { return cum[i]; });
  var bars = D.bars, barArr = bars.map(function (b) { return b[3]; });
  var allLL = lat.map(function (y, i) { return [y, lon[i]]; });

  function num(x, digits) { return x.toLocaleString(T.locale, { minimumFractionDigits: digits, maximumFractionDigits: digits }); }
  function fill(s, vars) { return s.replace(/\{(\w+)\}/g, function (_, k) { return vars[k]; }); }

  document.querySelectorAll('[data-t]').forEach(function (el) { el.textContent = T[el.getAttribute('data-t')]; });
  document.getElementById('scrub').setAttribute('aria-label', T.progress);
  document.getElementById('speed').setAttribute('aria-label', T.speed);

  var map = L.map('map', { preferCanvas: true, zoomSnap: 0.25, zoomControl: false, maxZoom: 19 });
  L.control.zoom({ position: 'bottomright', zoomInTitle: T.zoomIn, zoomOutTitle: T.zoomOut }).addTo(map);
  if (D.tiles.url) {
    L.tileLayer(D.tiles.url, { maxZoom: 19, maxNativeZoom: D.tiles.maxNativeZoom, attribution: D.tiles.attribution }).addTo(map);
  } else {
    map.attributionControl.addAttribution(D.tiles.attribution);   // no basemap: keep the data credits
  }
  map.fitBounds(L.latLngBounds(allLL).pad(0.05));

  map.createPane('lines'); map.getPane('lines').style.zIndex = 410;
  map.createPane('bars');  map.getPane('bars').style.zIndex = 420;
  map.createPane('head');  map.getPane('head').style.zIndex = 430;
  var rLines = L.canvas({ pane: 'lines' }), rBars = L.canvas({ pane: 'bars' }), rHead = L.canvas({ pane: 'head' });

  // bars: dim until reached
  var UNV = { radius: 2.5, fillColor: '#6b7280', fillOpacity: 0.55, weight: 0 };
  var VIS = { radius: 3.5, fillColor: '#ffd166', fillOpacity: 0.95, color: '#0e1116', weight: 1, opacity: 0.9 };   // dark ring keeps neighbours apart
  var markers = bars.map(function (b) {
    var m = L.circleMarker([b[1], b[0]], Object.assign({ renderer: rBars, pane: 'bars' }, UNV));
    if (b[2]) m.bindTooltip(b[2]);
    return m.addTo(map);
  });

  function legColor() { return '#3b9cff'; }
  var legs = new Array(nLegs), group = L.layerGroup().addTo(map), drawn = 0;
  function legLine(i) {
    if (!legs[i]) {
      var a = i === 0 ? 0 : legEnd[i - 1], b = legEnd[i];
      legs[i] = L.polyline(allLL.slice(a, b + 1), { renderer: rLines, pane: 'lines', color: legColor(i), weight: 3, opacity: 0.95, interactive: false });
    }
    return legs[i];
  }
  var partial = L.polyline([], { renderer: rLines, pane: 'lines', color: '#9fd0ff', weight: 3, opacity: 0.95, interactive: false }).addTo(map);
  var head = L.circleMarker(allLL[0], { renderer: rHead, pane: 'head', radius: 7, color: '#ffffff', weight: 2, fillColor: '#ff4d4d', fillOpacity: 1, interactive: false }).addTo(map);

  function upper(a, x) { var lo = 0, hi = a.length; while (lo < hi) { var m = (lo + hi) >> 1; if (a[m] <= x) lo = m + 1; else hi = m; } return lo; }

  var els = {};
  ['km', 'pct', 'totalkm', 'time', 'visited', 'nbars', 'scrub', 'play', 'restart', 'speed', 'speedlabel', 'follow', 'title', 'nextname', 'nextdist', 'toggle', 'panel']
    .forEach(function (id) { els[id] = document.getElementById(id); });

  // small screens start with a compact panel; the chevron expands it
  function setCompact(c) {
    els.panel.classList.toggle('compact', c);
    els.toggle.setAttribute('aria-expanded', String(!c));
    els.toggle.setAttribute('aria-label', c ? T.expand : T.collapse);
  }
  setCompact(window.matchMedia('(max-width: 640px)').matches);
  els.toggle.addEventListener('click', function () { setCompact(!els.panel.classList.contains('compact')); });
  els.title.textContent = D.title;
  els.totalkm.textContent = num(total / 1000, 1);
  els.nbars.textContent = bars.length;
  els.km.textContent = num(0, 1);

  function fmtTime(h) { return h >= 24 ? Math.floor(h / 24) + ' ' + T.days + ' ' + Math.floor(h % 24) + ' ' + T.hours : num(h, 1) + ' ' + T.hours; }
  function fmtDist(m) { return m < 1000 ? Math.round(m) + ' ' + T.m : num(m / 1000, 1) + ' ' + T.km; }

  // ring around the bar we are walking towards
  var ring = L.circleMarker([0, 0], { renderer: rHead, pane: 'head', radius: 11, color: '#ffd166', weight: 2, fill: false, interactive: false });

  function updateNext() {
    if (visited >= bars.length) {
      var finished = d >= total - 0.5;
      if (ring._map) map.removeLayer(ring);
      els.nextname.textContent = finished ? T.complete : T.backToStart;
      els.nextdist.textContent = finished ? '' : fill(T.toGo, { d: fmtDist(total - d) });
      return;
    }
    var a = barArr[visited], end = upper(barArr, a), names = [], i;     // bars sharing a stop have the same arrive_m
    for (i = visited; i < end && names.length < 3; i++) names.push(bars[i][2] || T.unnamed);
    var more = end - visited - names.length;
    els.nextname.textContent = names.join(', ') + (more > 0 ? ' ' + fill(T.more, { n: more }) : '');
    els.nextdist.textContent = fill(T.nextDist, { d: fmtDist(Math.max(0, a - d)) });
    ring.setLatLng([bars[visited][1], bars[visited][0]]);
    if (!ring._map) ring.addTo(map);
  }

  var d = 0, visited = 0;
  function setProgress(newD) {
    d = Math.max(0, Math.min(total, newD));
    var j = Math.min(Math.max(upper(cum, d) - 1, 0), cum.length - 1);
    var hp;
    if (j >= cum.length - 1) { hp = allLL[allLL.length - 1]; }
    else {
      var den = cum[j + 1] - cum[j], f = den > 0 ? (d - cum[j]) / den : 0;
      hp = [allLL[j][0] + (allLL[j + 1][0] - allLL[j][0]) * f, allLL[j][1] + (allLL[j + 1][1] - allLL[j][1]) * f];
    }
    var done = upper(legEndCum, d);
    while (drawn < done) { group.addLayer(legLine(drawn)); drawn++; }
    while (drawn > done) { drawn--; group.removeLayer(legLine(drawn)); }
    if (done < nLegs) {
      var s = done === 0 ? 0 : legEnd[done - 1];
      partial.setLatLngs(allLL.slice(s, Math.max(j, s) + 1).concat([hp]));
    } else { partial.setLatLngs([]); }
    head.setLatLng(hp);
    if (els.follow.checked) map.panTo(hp, { animate: false });

    var nv = upper(barArr, d), i;
    if (nv > visited) { for (i = visited; i < nv; i++) { markers[i].setStyle(VIS); markers[i].setRadius(VIS.radius); } }
    else if (nv < visited) { for (i = nv; i < visited; i++) { markers[i].setStyle(UNV); markers[i].setRadius(UNV.radius); } }
    visited = nv;

    els.km.textContent = num(d / 1000, 1);
    els.pct.textContent = num(100 * d / total, 0);
    els.time.textContent = fmtTime(d / 5000);
    els.visited.textContent = visited;
    updateNext();
    if (document.activeElement !== els.scrub) els.scrub.value = Math.round(1000 * d / total);
  }

  function speedKmPerS() { return 0.05 * Math.pow(10, Number(els.speed.value) / 50); }
  function showSpeed() {
    var s = speedKmPerS();
    els.speedlabel.textContent = fill(T.speedLabel, { s: num(s, s < 1 ? 2 : 1), n: Math.round(total / 1000 / s) });
  }
  var initial = (total / 1000) / 75;
  els.speed.value = Math.max(0, Math.min(150, Math.round(50 * Math.log10(initial / 0.05))));
  showSpeed();
  els.speed.addEventListener('input', showSpeed);

  var playing = false, last = 0;
  function frame(ts) {
    if (!playing) return;
    if (!last) last = ts;
    var dt = Math.min((ts - last) / 1000, 0.1); last = ts;
    setProgress(d + speedKmPerS() * 1000 * dt);
    if (d >= total) { pause(); return; }
    requestAnimationFrame(frame);
  }
  function play() { if (d >= total) setProgress(0); playing = true; last = 0; els.play.textContent = T.pause; requestAnimationFrame(frame); }
  function pause() { playing = false; els.play.textContent = d >= total ? T.replay : T.play; }
  els.play.textContent = T.play;
  els.play.addEventListener('click', function () { playing ? pause() : play(); });
  els.restart.addEventListener('click', function () { setProgress(0); if (!playing) els.play.textContent = T.play; });
  els.scrub.addEventListener('input', function () { setProgress(total * Number(els.scrub.value) / 1000); });
  els.follow.addEventListener('change', function () {
    if (els.follow.checked) map.setView(head.getLatLng(), 14, { animate: false });
    else map.fitBounds(L.latLngBounds(allLL).pad(0.05), { animate: false });
  });
  document.addEventListener('keydown', function (e) { if (e.code === 'Space' && e.target.tagName !== 'INPUT' && e.target.tagName !== 'BUTTON') { e.preventDefault(); playing ? pause() : play(); } });

  setProgress(0);
  window.liveMap = { setProgress: setProgress, total: total, play: play, pause: pause };
})();
</script>
</body>
</html>
"""

CARTO_ATTRIB = "&copy; OpenStreetMap contributors &copy; CARTO"
ESRI_ATTRIB = "Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"


def _basemap(carto_key, tile_url, attribution, max_native_zoom):
    """Pick the basemap: your own tile_url, else CARTO dark if you pass a key, else Esri dark gray (no key)."""
    if tile_url:
        return {"url": tile_url, "attribution": attribution or "", "maxNativeZoom": max_native_zoom or 19}
    if carto_key:
        return {"url": f"https://basemaps.cartocdn.com/rastertiles/dark_all/{{z}}/{{x}}/{{y}}.png?key={carto_key}",
                "attribution": CARTO_ATTRIB, "maxNativeZoom": 19}
    return {"url": "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
            "attribution": ESRI_ATTRIB, "maxNativeZoom": 16}


def make_live_map(route, bars, title=None, lang="en", carto_key=None, tile_url=None,
                  attribution=None, max_native_zoom=None, leaflet_css=LEAFLET_CSS, leaflet_js=LEAFLET_JS):
    t = STRINGS[lang]
    title = title or t["title"]
    route = route.sort_values("seq")
    leg = route["leg_no"].to_numpy()
    leg_end = np.flatnonzero(np.r_[leg[1:] != leg[:-1], True]).tolist()   # index of the last node of each leg

    cum = route["cum_m"].to_numpy(dtype=float)
    assert np.all(np.diff(cum) >= 0), "cum_m must be non-decreasing"

    b = bars.sort_values("arrive_m")
    bar_rows = [[round(float(x), 5), round(float(y), 5), "" if n is None else str(n), round(float(a), 1)]
                for x, y, n, a in zip(b["lon"], b["lat"], b["name"], b["arrive_m"])]

    data = {
        "title": title,
        "t": t,
        "tiles": _basemap(carto_key, tile_url, attribution, max_native_zoom),
        "route": {
            "lon": [round(float(v), 5) for v in route["lon"]],
            "lat": [round(float(v), 5) for v in route["lat"]],
            "cum": [round(float(v), 1) for v in cum],
            "legEnd": leg_end,
        },
        "bars": bar_rows,
    }
    return _render(data, lang, leaflet_css, leaflet_js)


def _render(data, lang, leaflet_css, leaflet_js):
    """Fill the page template with already-built data (also used to rebuild a page from saved data)."""
    payload = json.dumps(data, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    html = _TEMPLATE
    for key, val in (("__LEAFLET_CSS__", leaflet_css), ("__LEAFLET_JS__", leaflet_js),
                     ("__TITLE__", data["title"].replace("<", "&lt;")), ("__LANG__", lang)):
        html = html.replace(key, val)
    return html.replace("__DATA__", payload)

In [0]:
html = make_live_map(route, bars_live, title="Bars of St. Petersburg")
displayHTML(html)

with open("/Volumes/workspace/spb/landing/bar_crawl.html", "w", encoding="utf-8") as f:
    f.write(html)

In [0]:
df = spark.read.text("/Volumes/workspace/spb/landing/bar_crawl.html")
display(df)